# 01 — Exploratory Data Analysis
**Urban AQI Prediction | Berlin, Madrid, Paris, Rome**

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

ROOT = Path().resolve().parents[0]
UNZIPPED = ROOT / 'data/raw/eea/unzipped'
WEATHER = ROOT / 'data/raw/weather'

# find all parquet files in nested folders
files = list(UNZIPPED.glob('**/*.parquet'))
print(f"Found {len(files)} parquet files.")

Found 544 parquet files.


In [10]:
# load all files into a single dataframe
dfs = []
for f in files:
    tmp = pd.read_parquet(f)
    tmp['_source_file'] = f.name
    dfs.append(tmp)

df_eea = pd.concat(dfs, ignore_index=True)
print(f"Total EEA records: {df_eea.shape[0]:,}")

Total EEA records: 4,737,080


In [11]:
# automatically extract city based on known codes
def get_city(station: str) -> str:
    s = str(station)
    if 'DE' in s:
        return 'Berlin'
    elif 'ES' in s:
        return 'Madrid'
    # Automatically detecting FR and IT
    elif 'FR' in s:
        return 'Paris'
    elif 'IT' in s:
        return 'Rome'
    return 'Unknown'

df_eea['City'] = df_eea['Samplingpoint'].apply(get_city)
print("Measurements by city:")
print(df_eea['City'].value_counts())

Measurements by city:
City
Madrid    3112184
Berlin    1624896
Name: count, dtype: int64


In [12]:
# filter valid measurements only (1 or 2 means valid)
df_eea = df_eea[df_eea['Validity'].isin([1, 2])]
print(f"Valid records: {df_eea.shape[0]:,}")

Valid records: 4,528,802


In [13]:
# map pollutants
pol_map = {8: 'PM2.5', 7: 'PM10', 5: 'NO2', 6001: 'O3'}
df_eea['Pollutant_Name'] = df_eea['Pollutant'].map(pol_map)

# drop if not in our 4 targeted pollutants
df_eea = df_eea.dropna(subset=['Pollutant_Name'])
print(df_eea['Pollutant_Name'].value_counts())

Pollutant_Name
PM2.5    1649504
NO2      1121782
PM10      905900
O3        851616
Name: count, dtype: int64


In [14]:
# pivot the data so pollutants become columns
df_eea['Start'] = pd.to_datetime(df_eea['Start'])
# pivot table
df_pivoted = df_eea.pivot_table(
    index=['Start', 'City'], 
    columns='Pollutant_Name', 
    values='Value', 
    aggfunc='mean'
).reset_index()

print("Pivoted shape:", df_pivoted.shape)
df_pivoted.head()

Pivoted shape: (34937, 6)


Pollutant_Name,Start,City,NO2,O3,PM10,PM2.5
0,2022-01-01 00:00:00,Berlin,59.899167,47.848333,35.56125,13.99
1,2022-01-01 00:00:00,Madrid,30.8,25.5,8.579091,40.4375
2,2022-01-01 01:00:00,Berlin,23.55,19.285833,37.28125,11.107647
3,2022-01-01 01:00:00,Madrid,46.526316,38.833333,7.098182,43.625
4,2022-01-01 02:00:00,Berlin,17.518333,14.211667,38.985,9.126471


In [15]:
# load weather data and merge
weather_files = list(WEATHER.glob('*.csv'))

dfs_w = []
for f in weather_files:
    # file format: openmeteo_berlin_2022-01-01_2025-12-31.csv
    # extract city from filename
    city_name = f.stem.split('_')[1].capitalize()
    
    # skip rows to read actual csv headers from openmeteo (first 3 rows are meta/blank)
    w_df = pd.read_csv(f, skiprows=3)
    # Strip units from column names (e.g. 'temperature_2m (°C)' -> 'temperature_2m')
    w_df.columns = [c.split(' ')[0] for c in w_df.columns]
    w_df['City'] = city_name
    dfs_w.append(w_df)

df_weather = pd.concat(dfs_w, ignore_index=True)
df_weather['time'] = pd.to_datetime(df_weather['time'])
print("Weather data shape:", df_weather.shape)

Weather data shape: (140256, 7)


In [16]:
# Merge EEA and Weather
df_merged = pd.merge(
    df_pivoted, 
    df_weather, 
    left_on=['Start', 'City'], 
    right_on=['time', 'City'], 
    how='inner'
)
# clean up columns
df_merged = df_merged.drop(columns=['time'])

out_dir = ROOT / 'data/processed'
out_dir.mkdir(exist_ok=True)
df_merged.to_parquet(out_dir / 'merged_hourly_raw.parquet')
print("Saved to merged_hourly_raw.parquet")

Saved to merged_hourly_raw.parquet
